# Production plan with a distribution network

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ula-uab/lscm-decision-making/blob/main/cases/production_plan/notebooks/distribution_network.ipynb)

A company assembles electric bicycles in three plants and plans how many bikes each plant assembles each month from January to June. In the third version of the model, M2, the plants ship to a central warehouse, which holds stock and delivers the orders. With forecast 4, whose peak is in April and May, M2 has no solution: the bikes that have to be assembled in advance do not fit in the plants and the central warehouse. This notebook uses the fourth version, M3: a distribution network in Spain with the plants, the central warehouse, a rail terminal, two regional warehouses and four local warehouses, which receive the orders. The network is a list of arcs, each with a transport cost and an arc capacity.

Contents of the notebook:

1. Network, nodes and forecast 4.
2. Linear model of the distribution network.
3. Optimal plan under forecast 4.
4. Three tests, each with one datum changed: the arc capacity of arc 6 (rail), the transport rate of rail and the node capacity of regional warehouse 2.

The cells run in order, with ▶ or *Shift + Enter*. Some cells have a form (sliders, drop-down lists and tick boxes): after changing a value, run the cell again. Outside Colab the form is a line such as `month = "May"`: edit the value and run the cell.

The table gives the symbols of M0 to M3, with the term that names each concept. The column «From» gives the version where each symbol first appears; in M3 the warehouses are nodes $i$ of the network.

| Symbol | Meaning | Term | Unit | From |
|---|---|---|---|---|
| $F$, $f$ | Plants | — | — | M0 |
| $T$, $t$ | Months, January to June; $t = 0$ is the end of December | — | — | M0 |
| $\tau$ | Last month of the horizon | — | — | M1 |
| $V$, $i$ | Nodes of the network: plants, warehouses and the rail terminal | — | — | M3 |
| $A$, $a$ | Arcs of the network; the arc $a = \langle i, j \rangle$ goes from its origin node $i$ to its destination node $j$ | — | — | M3 |
| $A^{\text{in}}_j$; $A^{\text{out}}_i$ | Arcs that enter node $j$; arcs that leave node $i$ | — | — | M3 |
| $V^d$ | Demand nodes: the local warehouses | demand node | — | M3 |
| $c_f$ | Unit assembly cost at plant $f$ | — | €/bike | M0 |
| $K_{ft}$ | Assembly capacity of plant $f$ in month $t$ | — | bikes/month | M0 |
| $d_{it}$ | Orders that node $i$ receives in month $t$ | — | bikes/month | M3 |
| $c^h_i$ | Holding cost at node $i$ | holding cost | €/(bike·month) | M1 |
| $h_{i0}$ | Initial stock at node $i$, at the end of December | — | bikes | M1 |
| $h^{\text{end}}_i$ | Stock that node $i$ must hold at the end of the horizon | required final stock | bikes | M1 |
| $K^h_i$ | Units node $i$ can hold | storage capacity | bikes | M2 |
| $c^b_i$ | Cost of one bike backordered for one month at node $i$ | backorder cost | €/(bike·month) | M2 |
| $b_{i0}$; $\bar b_i$ | Backorders at node $i$ at the end of December; maximum at the end of the horizon | backorders | bikes | M2 |
| $c^x_a$ | Cost of carrying one bike along arc $a$ | transport cost | €/bike | M3 |
| $K^x_a$ | Maximum flow on arc $a$ in a month | arc capacity | bikes/month | M3 |
| $K^n_i$ | Maximum flow that can enter node $i$ in a month | node capacity | bikes/month | M3 |
| $p_{ft}$ | Bikes assembled at plant $f$ in month $t$ | — | bikes/month | M0 |
| $h_{it}$ | Bikes in stock at node $i$ at the end of month $t$ | — | bikes | M1 |
| $x_{at}$ | Bikes shipped on arc $a$ in month $t$ | shipments | bikes/month | M1 |
| $s_{it}$ | Bikes that node $i$ delivers in month $t$ to orders of that month or to backorders | deliveries | bikes/month | M2 |
| $b_{it}$ | Orders received by node $i$ and not yet delivered at the end of month $t$ | backorders | bikes | M2 |

The [document of this version](https://github.com/ula-uab/lscm-decision-making/blob/main/cases/production_plan/distribution_network.md) explains all the data and results. The places and distances are real; the other data are invented. Running the notebook is optional: it is support material, not part of the assessment.

In [ ]:
# @title Setup { display-mode: "form" }
# In Google Colab, install the example from GitHub (it takes a few seconds).
# On your own computer, install the repository environment instead (see README).
import sys
if "google.colab" in sys.modules:
    url = "git+https://github.com/ula-uab/lscm-decision-making#subdirectory=cases/production_plan"
    %pip install -q {url}
    # A session that already had the example keeps its old code unless it is reinstalled
    %pip install -q --force-reinstall --no-deps {url}
    # and unless its old code is removed from memory, so that the import below reads
    # the new one: no need to restart the session
    import importlib
    for name in [m for m in sys.modules if m == "plants" or m.startswith("plants.")]:
        del sys.modules[name]
    importlib.invalidate_caches()
from plants import distribution_network as dn

## 1. Network, nodes and forecast 4

The cell gives the transport rate of each mode and three figures. The map places each node of the network in its city. The second figure draws the network by stages, from the plants on the left to the local warehouses on the right, with the data of each node under its name: for a plant, its unit assembly cost and capacity, its holding cost and storage capacity, and its initial stock and required final stock; for a warehouse, its holding cost and storage capacity and, for regional warehouse 2, its node capacity, the bikes it can receive in a month. The rail terminal holds no stock. On each arc, its transport cost, its distance times the transport rate of its mode, and its arc capacity; the rail arc is dashed. The third figure gives the orders of each local warehouse in each month: a share of forecast 4, the same every month.

`show_tables`: instead of the figures, four tables: the place of each node, with its latitude and longitude; the list of arcs, with the transport mode and the distance of each one; the data of each node; and the orders of each local warehouse.

By which routes can bikes reach regional warehouse 2 from the central warehouse, and what is the transport cost of one bike on each route?

In [ ]:
# @title Network, nodes and forecast 4 { display-mode: "form" }
show_tables = False  # @param {type:"boolean"}
dn.show_data(show_tables=show_tables)

## 2. Linear model of the distribution network

$$
\begin{aligned}
\min \quad & \sum_{t \in T} \Big( \sum_{f \in F} c_f\, p_{ft} + \sum_{i \in V} c^h_i\, h_{it} + \sum_{a \in A} c^x_a\, x_{at} + \sum_{i \in V^d} c^b_i\, b_{it} \Big) \\
\text{s.t.} \quad & h_{i,t-1} + p_{it} + \sum_{a \in A^{\text{in}}_i} x_{at} = \sum_{a \in A^{\text{out}}_i} x_{at} + s_{it} + h_{it} && \forall i \in V,\ t \in T && \text{(stock balance at each node)} \\
& \sum_{a \in A^{\text{out}}_i} x_{at} + s_{it} \le h_{i,t-1} + p_{it} + \sum_{a \in A^{\text{in}}_i} x_{at} && \forall i \in V,\ t \in T && \text{(shipment limit at each node)} \\
& b_{i,t-1} + d_{it} = s_{it} + b_{it} && \forall i \in V^d,\ t \in T && \text{(backorder balance)} \\
& x_{at} \le K^x_a && \forall a \in A,\ t \in T && \text{(arc capacity)} \\
& \sum_{a \in A^{\text{in}}_i} x_{at} \le K^n_i && \forall i \text{ with a node capacity},\ t \in T && \text{(node capacity)} \\
& h_{it} \le K^h_i && \forall i \in V,\ t \in T && \text{(storage capacity)} \\
& p_{ft} \le K_{ft} && \forall f \in F,\ t \in T && \text{(assembly capacity)} \\
& b_{i\tau} \le \bar b_i \quad \forall i \in V^d, \qquad h_{i\tau} \ge h^{\text{end}}_i \quad \forall i \in V && && \text{(backorders and required final stock at the end)} \\
& p_{ft},\ h_{it},\ x_{at},\ s_{it},\ b_{it} \ge 0
\end{aligned}
$$

with $h_{i0}$ and $b_{i0}$ given, $p_{it} = 0$ if node $i$ is not a plant and $s_{it} = 0$ if node $i$ is not a demand node. One stock balance holds at every node: what the node held at the start of the month, plus what it assembles and what arrives on the arcs that enter it, equals what leaves on the arcs that leave it, plus its deliveries, plus what it holds at the end of the month. Only the plants assemble and only the local warehouses deliver; at the rail terminal both are 0, and its storage capacity is 0. There is one flow variable for each arc of the list.

## 3. Optimal plan under forecast 4

The cell first gives the reason why M2 has no solution under forecast 4. Then it solves M3 and gives the cost of the plan and three figures. The first draws the network with the flow on each arc in the month of the form: an arc whose capacity is full, used entirely in that month, is red, and an arc with no flow is light grey; under each node, its stock at the end of the month and, at each local warehouse, also its deliveries in the month and its backorders at the end of the month. The second gives the bikes that each plant assembles in each month against its capacity (dashed line). The third has two panels: the stock at each stage of the network at the end of each month, stacked, and the **shadow price** of each capacity that is full in the month of the form (€/bike): the change in the cost of the plan with one more bike of that capacity in that month (negative: a saving).

`month`: the month of the network and of the shadow prices.

`show_tables`: instead of the figures, five tables: the bikes assembled at each plant, the flow on each arc, the stock of each node at the end of each month, the deliveries and the backorders of the local warehouses, and the shadow prices of the capacities that are full in some month.

Choose May. By which routes do bikes reach regional warehouse 2, and which arcs are full? Is the rail arc full? In which months is there stock at more than one stage of the network, and why then?

In [ ]:
# @title Optimal plan under forecast 4 { display-mode: "form" }
month = "May"  # @param ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]
show_tables = False  # @param {type:"boolean"}
dn.network_plan(month, show_tables=show_tables);

## 4. Arc capacity of arc 6 (rail)

Arc 6 is the rail arc, from the dry port of Zaragoza to the rail terminal in Coslada. The rail terminal holds no stock, so the rail arc carries only what the road arc from the terminal to regional warehouse 2 carries, at most 200 bikes a month. The cell gives the cost of the plan and, first, a figure of the flow in each month on the three routes to regional warehouse 2 and on the two routes to Valencia; then the first two figures of section 3 and the stock at each stage, with the flows of May on the network.

`rail_arc_capacity`: the arc capacity of arc 6, in bikes/month.

`show_tables`: instead of the figures, the flows on the five routes in a table, and the tables of section 3 without the shadow prices.

Lower the arc capacity of arc 6 from 400. From what value does the rail arc fill, and which routes carry the bikes that no longer fit? Below what value is there no solution?

In [ ]:
# @title Optimal plan for an arc capacity of arc 6 { display-mode: "form" }
rail_arc_capacity = 400  # @param {type:"slider", min:0, max:400, step:10}
show_tables = False  # @param {type:"boolean"}
dn.rail_capacity_test(rail_arc_capacity, show_tables=show_tables);

## 5. Transport rate of rail

The route by rail and the rail terminal competes with the road from the central warehouse to regional warehouse 2, whose transport cost is 3,30 €/bike and whose arc capacity is 150 bikes a month. The cell shows the same figures as the previous one.

`rail_transport_rate`: the transport rate of rail, in €/(bike·km); the transport cost of the rail arc is its distance, 336 km, times that rate.

`show_tables`: instead of the figures, the flows on the five routes in a table, and the tables of section 3 without the shadow prices.

Raise the transport rate of rail from 0,006. At what rate do the two routes cost the same, and what happens at exactly that rate? Above it, does the rail arc carry nothing, or what does it carry?

In [ ]:
# @title Optimal plan for a transport rate of rail { display-mode: "form" }
rail_transport_rate = 0.006  # @param {type:"slider", min:0.002, max:0.015, step:0.0005}
show_tables = False  # @param {type:"boolean"}
dn.rail_rate_test(rail_transport_rate, show_tables=show_tables);

## 6. Node capacity of regional warehouse 2

The node capacity of regional warehouse 2 is the bikes it can receive in a month, by any of its three routes. Regional warehouse 2 supplies Seville alone and shares Valencia with regional warehouse 1. The cell shows the same figures as the previous ones.

`node_capacity`: the node capacity of regional warehouse 2, in bikes/month.

`show_tables`: instead of the figures, the flows on the five routes in a table, and the tables of section 3 without the shadow prices.

Raise the node capacity from 280, then lower it. When is it full, and where do the bikes that it cannot receive go? Below what value is there no solution, and why?

In [ ]:
# @title Optimal plan for a node capacity of regional warehouse 2 { display-mode: "form" }
node_capacity = 280  # @param {type:"slider", min:150, max:400, step:10}
show_tables = False  # @param {type:"boolean"}
dn.node_capacity_test(node_capacity, show_tables=show_tables);

The [document of this version](https://github.com/ula-uab/lscm-decision-making/blob/main/cases/production_plan/distribution_network.md) explains all the numbers and has a script that prints them.